In [2]:
import numpy as np
import os
from PIL import Image
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

def load_devnagari_data(data_dir):
    images = []
    # Assuming the dataset has folders named '0', '1', etc.
    for digit in range(10):
        digit_path = os.path.join(data_dir, str(digit))
        for img_name in os.listdir(digit_path):
            img = Image.open(os.path.join(digit_path, img_name)).convert('L') # Load using PIL
            img = img.resize((28, 28))
            images.append(np.asarray(img))

    # Normalize to [0, 1] [cite: 64]
    x = np.array(images).astype('float32') / 255.0
    # Reshape for Keras compatibility (channel dimension) [cite: 64]
    x = np.reshape(x, (len(x), 28, 28, 1))
    return x

# Replace 'path_to_your_dataset' with your actual folder path
# x_data = load_devnagari_data('/content/drive/MyDrive/AI & ML/week7/data.csv')

# Split into training and validation [cite: 65]
# x_train, x_val = train_test_split(x_data, test_size=0.2, random_state=42)

# For demonstration, generating dummy data shaped like the dataset
x_train = np.random.random((1000, 28, 28, 1))
x_val = np.random.random((200, 28, 28, 1))

# Add Gaussian noise with a factor of 0.5 [cite: 33, 65]
noise_factor = 0.5
x_train_noisy = x_train + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_train.shape)
x_val_noisy = x_val + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_val.shape)

# Clip to ensure values remain [0, 1] [cite: 34]
x_train_noisy = np.clip(x_train_noisy, 0., 1.)
x_val_noisy = np.clip(x_val_noisy, 0., 1.)

In [5]:
from tensorflow.keras import layers, models

# 2. Build the Denoising Convolutional Autoencoder
input_img = layers.Input(shape=(28, 28, 1))

# Encoder
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
x = layers.MaxPooling2D((2, 2), padding='same')(x)
x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(x)
encoded = layers.MaxPooling2D((2, 2), padding='same')(x)

# Decoder
x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(encoded)
x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = layers.UpSampling2D((2, 2))(x)
decoded = layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

autoencoder = models.Model(input_img, decoded)
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

In [6]:
# 3. Train the Model
history = autoencoder.fit(x_train_noisy, x_train,
                epochs=50,
                batch_size=128,
                shuffle=True,
                validation_split=0.2)

# Plot Loss Curves
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.legend()
plt.show()

NameError: name 'x_train_noisy' is not defined

In [9]:
# 4. Visualize Results
decoded_imgs = autoencoder.predict(x_train_noisy)

n = 5  # Number of digits to display
plt.figure(figsize=(10, 4))
for i in range(n):
    # Original
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(x_train[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    # Noisy
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(x_train_noisy[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    # Denoised (Reconstructed)
    ax = plt.subplot(3, n, i + 1 + 2*n)
    plt.imshow(decoded_imgs[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
plt.show()

NameError: name 'x_train_noisy' is not defined